# 02_ex_agreement_topic — exploration and proposal notebook


## 01 Introduction and scope

- `agreement_id`: `<agreement_id>`
- `topic`: `<topic>`
- Source being explored: `<source_table or source_file_path>`
- Question being answered: `<business question>`
- Approved usage context: link to `01_data_sharing_agreement_<agreement>` decisions.


## 02 Agreement context

Capture approved usage boundaries, privacy constraints, and governance context that apply during exploration.

> `02_ex` proposes and captures evidence. It does **not** approve governance controls and does **not** enforce production DQ rules.


## 03 Configuration and setup


In [ ]:
%run 00_env_config

from fabricops_kit import get_path, setup_fabricops_notebook

agreement_id = "<agreement_id>"
topic = "<topic>"
source_env = ENV_NAME
source_lakehouse = "<source_lakehouse>"
source_warehouse = "<source_warehouse>"
source_table = "<source_table>"
source_file_path = "<source_file_path>"
profile_output_table = "<profile_output_table>"
proposal_output_table = "<proposal_output_table>"
metadata_evidence_table = "<metadata_evidence_table>"

setup = setup_fabricops_notebook(config=FABRIC_CONFIG, env_name=ENV_NAME)
metadata_path = get_path(ENV_NAME, "metadata", config=FABRIC_CONFIG)
print("Environment:", source_env)
print("Metadata path:", metadata_path)


In [ ]:
import re

notebook_name = "02_ex_<agreement>_<topic>"  # Replace with current notebook filename if available.
if not re.match(r"^02_ex_[a-z0-9_]+_[a-z0-9_]+$", notebook_name):
    raise ValueError("Notebook name must follow 02_ex_<agreement>_<topic>.")


## 04 Data ingestion for exploration


In [ ]:
from fabricops_kit import lakehouse_table_read, warehouse_read

# Choose one source path for exploration.
# Option A: table in lakehouse
try:
    df_source = lakehouse_table_read(source_lakehouse, source_table)
except Exception:
    # Option B: warehouse fallback
    df_source = warehouse_read(source_warehouse, source_table)

print("Row count:", df_source.count())
df_source.printSchema()


## 05 Source profiling


In [ ]:
from fabricops_kit import profile_dataframe

profile_df = profile_dataframe(df_source)
profile_df.show(50, truncate=False)

# Optional: persist profiling evidence for handoff.
# profile_df.write.mode("append").saveAsTable(profile_output_table)


## 06 Exploratory analysis


In [ ]:
# Inspect source quirks and analyst observations.
df_source.show(20, truncate=False)

# Example checks
# - Null concentration by key columns
# - Distinct counts
# - Min/max date spans


## 07 Exploratory transforms


In [ ]:
# Exploratory transform placeholder (non-production).
# These tests help define future 03_pc contract logic.

exploratory_df = (
    df_source
    # .filter("status IS NOT NULL")
    # .withColumn("event_date", F.to_date("event_ts"))
)

exploratory_df.show(20, truncate=False)


## 08 AI-assisted DQ and classification proposals


In [ ]:
from fabricops_kit import draft_dq_rules

# Optional AI suggestion placeholder.
dq_candidates = draft_dq_rules(
    profile_df=profile_df,
    table_name=source_table,
    business_context="<business_context_summary>",
)

display(dq_candidates)

# Optional: add classification/sensitivity candidate workflow if enabled in your environment.


## 09 Findings and proposed metadata updates


In [ ]:
# Capture proposed findings/evidence rows for governance review.
proposal_rows = [
    {
        "agreement_id": agreement_id,
        "topic": topic,
        "source_table": source_table,
        "proposal_type": "dq_rule_candidate",
        "proposal_summary": "<candidate rule and rationale>",
        "status": "proposed",
    }
]

proposal_evidence_df = spark.createDataFrame(proposal_rows)
proposal_evidence_df.show(truncate=False)

# Optional persistence
# proposal_evidence_df.write.mode("append").saveAsTable(proposal_output_table)
# proposal_evidence_df.write.mode("append").saveAsTable(metadata_evidence_table)


## 10 Handoff to governance and pipeline contract

- Governance updates go to `01_data_sharing_agreement_<agreement>` for human approval.
- Approved rules/classifications are loaded later by `03_pc_<agreement>_<pipeline>`.
- Final repeatable run-all-safe transforms and enforcement belong in `03_pc`.


## 11 Frozen diagnostics / appendix


In [ ]:
# Freeze reproducibility diagnostics.
print({
    "agreement_id": agreement_id,
    "topic": topic,
    "source_env": source_env,
    "source_lakehouse": source_lakehouse,
    "source_warehouse": source_warehouse,
    "source_table": source_table,
    "source_file_path": source_file_path,
    "profile_output_table": profile_output_table,
    "proposal_output_table": proposal_output_table,
    "metadata_evidence_table": metadata_evidence_table,
})
